# LeetCode 379 — Design Phone Directory bằng Python

Link đề: https://leetcode.com/problems/design-phone-directory/ · Độ khó: **Medium** · Chủ đề: Design, Queue, Array, Hash Table, Linked List

**Bài Premium (🔒).** Ghi chú: đề được viết theo hiểu biết về bài này (không có tài khoản Premium để xem trực tiếp) — nếu bạn có Premium, hãy đối chiếu lại với đề gốc, đặc biệt các câu chữ chính xác và các ví dụ bổ sung. Cấu trúc lớp, tên method và ràng buộc dưới đây khớp với bản mô tả công khai của bài.

Skeleton LeetCode:

```python
class PhoneDirectory:

    def __init__(self, maxNumbers: int):
        ...

    def get(self) -> int:
        ...

    def check(self, number: int) -> bool:
        ...

    def release(self, number: int) -> None:
        ...


# Your PhoneDirectory object will be instantiated and called as such:
# obj = PhoneDirectory(maxNumbers)
# param_1 = obj.get()
# param_2 = obj.check(number)
# obj.release(number)
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề **thiết kế lớp**: một "danh bạ số điện thoại" có `maxNumbers` ô số `0 .. maxNumbers-1`; cấp một số chưa ai dùng (`get`), kiểm tra một số còn trống không (`check`), trả một số về kho (`release`).
2. Nắm các công cụ Python cho bài thiết kế: lớp với `__init__`/`self`, `set` (`add`, `remove`, `discard`, `pop`, `in`), `collections.deque` (`popleft`, `append`), danh sách cờ `bool`, thao tác **bit** trên số nguyên (`&`, `|`, `^`, `<<`, `x & -x`, `bit_length`), và cách **mô phỏng** cách LeetCode gọi lớp theo chuỗi lệnh.
3. Đi từ **mảng cờ + quét tuyến tính (`get` chậm `O(n)`)** → **tập `set` các số còn trống (`O(1)`)** → **hàng đợi + cờ chống trùng** → **bộ đếm `next` + ngăn xếp số đã trả (khởi tạo lười, `O(1)` bộ nhớ ban đầu)** → **bitmask dùng số nguyên lớn**.
4. Hiểu mấu chốt: ta phải đảm bảo **mỗi số hoặc "đang trống" hoặc "đang được cấp"** (bất biến), và **mọi thao tác giữ bất biến đó** — đặc biệt khi `release` một số **đã trống** hoặc gọi `get` khi kho **hết** số.
5. Tránh lỗi kinh điển: `release` số đã trống làm nhân đôi (cấp hai lần cùng một số), quên đánh dấu "đã cấp" trong `get`, sửa `set` khi đang duyệt, dùng `list.pop(0)` thay `deque.popleft()`, quên trả `-1` khi hết, và test sai vì giả định `get` trả đúng số nhỏ nhất.

## 2. Đề bài & ví dụ

Thiết kế một **danh bạ số điện thoại** ban đầu có `maxNumbers` ô trống, mỗi ô có thể chứa một số (các số có giá trị `0, 1, ..., maxNumbers - 1`). Danh bạ cần hỗ trợ cấp phát số, kiểm tra ô có trống không, và giải phóng ô.

Cài đặt lớp `PhoneDirectory`:

- `PhoneDirectory(int maxNumbers)`: khởi tạo danh bạ với `maxNumbers` ô có sẵn.
- `int get()`: cấp một số **chưa được gán cho ai**; trả `-1` nếu **không còn số nào**. (Có thể trả **bất kỳ** số trống nào; số vừa cấp trở thành "đã dùng".)
- `bool check(int number)`: trả `true` nếu ô `number` **đang trống**, ngược lại `false`.
- `void release(int number)`: **thu hồi** ô `number` (trả về kho để cấp lại).

**Ví dụ:**

```
Input
["PhoneDirectory", "get", "get", "check", "get", "check", "release", "check"]
[[3], [], [], [2], [], [2], [2], [2]]
Output
[null, 0, 1, true, 2, false, null, true]
```

Giải thích: `PhoneDirectory(3)`; `get()` → (giả sử) `0`; `get()` → `1`; `check(2)` → `true`; `get()` → `2` (số còn lại duy nhất); `check(2)` → `false`; `release(2)`; `check(2)` → `true`. (Vì `get` được phép trả **bất kỳ** số trống, một lời giải đúng có thể trả thứ tự khác, miễn nhất quán.)

**Ràng buộc:** `1 <= maxNumbers <= 10^4`; `0 <= number < maxNumbers`; tối đa `2 * 10^4` lần gọi tới `get`, `check`, `release`.

Ví dụ trên được dựng theo hiểu biết về đề (Premium) — nên đối chiếu lại đề gốc nếu có tài khoản.

In [1]:
import random
import time
from collections import deque
from typing import List, Optional

# Bộ chạy mô phỏng cách LeetCode gọi: names là danh sách tên lệnh, args là danh sách tham số tương ứng.
def run_commands(cls, names, args):
    out, obj = [], None
    for i, (name, a) in enumerate(zip(names, args)):
        if i == 0:                           # lệnh đầu tiên luôn là constructor (tên lớp)
            obj = cls(*a)
            out.append(None)                 # constructor -> null
        else:
            out.append(getattr(obj, name)(*a))
    return out

example_names = ["PhoneDirectory", "get", "get", "check", "get", "check", "release", "check"]
example_args = [[3], [], [], [2], [], [2], [2], [2]]
example_output = [None, 0, 1, True, 2, False, None, True]
print(list(zip(example_names, example_args, example_output)))

[('PhoneDirectory', [3], None), ('get', [], 0), ('get', [], 1), ('check', [2], True), ('get', [], 2), ('check', [2], False), ('release', [2], None), ('check', [2], True)]


## 3. Kiến thức nền

### 3.1. Lớp, `__init__` và `self` (bài thiết kế)

Bài "Design" yêu cầu viết một **lớp (class)**: một "hộp" gom **dữ liệu** (thuộc tính) và **hành động** (method). Khi LeetCode chạy, nó tạo một đối tượng `obj = PhoneDirectory(maxNumbers)` (gọi `__init__`) rồi gọi `obj.get()`, `obj.check(n)`, ... **nhiều lần trên cùng đối tượng**. Dữ liệu cần nhớ giữa các lần gọi phải lưu vào `self.xxx`.

In [2]:
class DemBoDem:
    def __init__(self, bat_dau):
        self.so = bat_dau            # thuộc tính: nhớ trạng thái giữa các lần gọi

    def tang(self):
        self.so += 1
        return self.so

b = DemBoDem(10)
print(b.tang(), b.tang(), b.tang())      # 11 12 13: cùng đối tượng giữ trạng thái
c = DemBoDem(0)
print(c.tang(), "| b.so =", b.so)         # c độc lập với b

11 12 13
1 | b.so = 13


### 3.2. Gọi method theo **tên chuỗi** — cách LeetCode chạy

LeetCode cho hai danh sách: `names` (tên lệnh) và `args` (tham số). `getattr(obj, "get")` trả về method `obj.get`; `getattr(obj, name)(*a)` gọi nó với tham số trong list `a` (`*a` "bung" list thành các tham số). Hàm `run_commands` ở trên làm đúng việc đó. Thử với lớp đồ chơi:

In [3]:
class Toy:
    def __init__(self, n):
        self.n = n
    def double(self):
        return self.n * 2
    def add(self, x):
        self.n += x

print(run_commands(Toy, ["Toy", "double", "add", "double"], [[5], [], [3], []]))     # [None, 10, None, 16]

[None, 10, None, 16]


### 3.3. `set` (tập hợp): thêm, xóa, kiểm tra thuộc, lấy một phần tử

`set` lưu các phần tử **không trùng, không có thứ tự**, và các thao tác chính là `O(1)` trung bình:

- `s.add(x)` thêm `x` (đã có thì không đổi);
- `s.remove(x)` xóa `x` (**lỗi `KeyError`** nếu không có); `s.discard(x)` xóa nếu có (không lỗi);
- `x in s` kiểm tra thuộc;
- `s.pop()` lấy và xóa **một** phần tử bất kỳ (lỗi nếu rỗng).

In [4]:
s = {3, 5, 8}
s.add(5); s.add(9)
print(s, 5 in s, 7 in s)
s.discard(100)                          # không lỗi dù không có
print(s.pop() in {3, 5, 8, 9}, len(s))  # pop lấy phần tử bất kỳ
try:
    s.remove(100)
except KeyError as e:
    print("remove số không có -> KeyError:", e)

{8, 9, 3, 5} True False
True 3
remove số không có -> KeyError: 100


### 3.4. `deque` (hàng đợi hai đầu)

`collections.deque` cho `append` (thêm cuối) và `popleft` (lấy đầu) đều `O(1)`. Với `list`, `pop(0)` là `O(n)` (phải dịch cả list). Hàng đợi (FIFO — vào trước ra trước) là cấu trúc tự nhiên cho "kho số": số nào trả về trước thì cấp lại trước.

In [5]:
q = deque([0, 1, 2])
print(q.popleft(), q.popleft())          # 0 1
q.append(7)
print(list(q))                           # [2, 7]
print(len(q), bool(q), bool(deque()))    # 2 True False   (deque rỗng -> False)

0 1
[2, 7]
2 True False


### 3.5. Thao tác bit trên số nguyên

Một số nguyên có thể dùng như **dãy bit**: bit `i` = `1` nghĩa là "số `i` còn trống".

- `1 << i`: số chỉ có bit `i` bằng `1`.
- `mask & (1 << i)`: kiểm tra bit `i`; `mask | (1 << i)`: bật bit `i`; `mask ^ (1 << i)`: đảo bit `i`.
- `(1 << n) - 1`: `n` bit toàn `1`.
- `mask & -mask`: **tách bit `1` thấp nhất** (số trống nhỏ nhất); `.bit_length() - 1` đổi nó thành chỉ số.

Python có số nguyên **lớn tùy ý**, nên `maxNumbers = 10^4` bit không vấn đề.

In [6]:
mask = (1 << 5) - 1                     # 0b11111: các số 0..4 đều trống
print(bin(mask))
thap_nhat = mask & -mask
print(bin(thap_nhat), thap_nhat.bit_length() - 1)   # 0b1 -> chỉ số 0
mask ^= thap_nhat                       # "cấp" số 0: tắt bit 0
print(bin(mask), "| bit 3 bật?", bool(mask >> 3 & 1))
mask |= 1 << 0                          # "trả" số 0: bật lại bit 0
print(bin(mask))
mask2 = 0b11000
print((mask2 & -mask2).bit_length() - 1)            # 3: bit 1 thấp nhất ở vị trí 3

0b11111
0b1 0
0b11110 | bit 3 bật? True
0b11111
3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hình dung một **bãi đỗ xe có `maxNumbers` ô** đánh số `0..n-1`. Mỗi ô ở một trong **hai trạng thái**: **trống** hay **đã cấp**.

- `get()`: tìm **một ô trống**, đánh dấu "đã cấp" và trả số ô; không còn ô nào → `-1`.
- `check(i)`: ô `i` có trống không.
- `release(i)`: đánh dấu ô `i` là trống.

Tính tay với `n = 3` theo ví dụ (ô trống = `T`, đã cấp = `X`):

| bước | lệnh | trạng thái các ô `[0,1,2]` | kết quả |
|---|---|---|---|
| 0 | `PhoneDirectory(3)` | `[T,T,T]` | null |
| 1 | `get()` | cấp ô 0 → `[X,T,T]` | 0 |
| 2 | `get()` | cấp ô 1 → `[X,X,T]` | 1 |
| 3 | `check(2)` | ô 2 trống | true |
| 4 | `get()` | cấp ô 2 → `[X,X,X]` | 2 |
| 5 | `check(2)` | ô 2 đã cấp | false |
| 6 | `release(2)` | trả ô 2 → `[X,X,T]` | null |
| 7 | `check(2)` | ô 2 trống | true |

Có thêm tình huống **góc cạnh** cần nghĩ trước khi code:

- `get()` khi `[X,X,X]` → `-1`.
- `release(i)` khi ô `i` **đã trống** → không làm gì (không được "trống hai lần").
- `release(i)` rồi `get()`: số `i` có thể được cấp lại.

Nút thắt thiết kế: làm sao `get()` tìm "một ô trống" **nhanh** mà không phải duyệt tất cả?

In [7]:
# Thí nghiệm giấy-bút bằng list cờ: True = trống
o = [True, True, True]
def get_tay(o):
    for i, trong in enumerate(o):
        if trong:
            o[i] = False
            return i
    return -1

print(get_tay(o), get_tay(o), o[2])            # 0 1 True
print(get_tay(o), o[2])                        # 2 False
print(get_tay(o))                              # -1 (hết số)
o[2] = True                                    # release(2)
print(o)

0 1 True
2 False
-1
[False, False, True]


## 5. Cách 1 — Brute Force / baseline: mảng cờ + quét tuyến tính

**Ý tưởng:** một list `is_free` độ dài `maxNumbers`, `is_free[i] = True` nếu số `i` còn trống. `get` **quét từ đầu** tìm số trống đầu tiên; `check`, `release` truy cập trực tiếp `is_free[number]`.

Code (đặt tên `PhoneDirectoryScan`):

In [8]:
class PhoneDirectoryScan:
    def __init__(self, maxNumbers: int):
        self.is_free = [True] * maxNumbers

    def get(self) -> int:
        for i in range(len(self.is_free)):       # quét từ đầu tìm số trống
            if self.is_free[i]:
                self.is_free[i] = False
                return i
        return -1

    def check(self, number: int) -> bool:
        return self.is_free[number]

    def release(self, number: int) -> None:
        self.is_free[number] = True

out = run_commands(PhoneDirectoryScan, example_names, example_args)
print(out)
assert out == example_output          # cách này luôn cấp số nhỏ nhất -> khớp đúng ví dụ

[None, 0, 1, True, 2, False, None, True]


**Độ phức tạp:** `check`, `release` `O(1)`. `get` **`O(n)`** trong trường hợp xấu (quét qua nhiều số đã cấp). Bộ nhớ `O(n)`.

**Nút thắt:** sau nhiều lần `get`, các ô đầu danh sách đều đã cấp; mỗi `get` mới phải **quét lại** qua chúng. Với `n = 10^4` và `10^4` lệnh `get`, tổng khoảng `n²/2 = 5·10^7` bước — chậm. Ta đo thời gian thấy rõ độ cong bậc hai:

In [9]:
def do_get_lien_tiep(cls, n):
    d = cls(n)
    t = time.perf_counter()
    for _ in range(n):
        d.get()
    return 1000 * (time.perf_counter() - t)

for n in (500, 1000, 2000, 4000):
    print(f"n = {n:>4}: {do_get_lien_tiep(PhoneDirectoryScan, n):8.2f} ms")

n =  500:     2.14 ms
n = 1000:     9.32 ms
n = 2000:    39.31 ms


n = 4000:   153.88 ms


Thời gian tăng khoảng **3–4 lần** mỗi khi `n` gấp đôi (lý thuyết: gấp 4): đúng dấu hiệu `O(n²)`. Ta cần `get` nhanh hơn.

## 6. Ý tưởng thuật toán / bước đột phá

1. **Đừng đi tìm ô trống — hãy *giữ sẵn danh sách các ô trống*.** Dùng một cấu trúc chứa **đúng** các số đang trống: `get` chỉ cần **lấy ra một phần tử** từ đó (không quét), `release` **thêm vào**, `check` hỏi "có trong cấu trúc không".
2. Cấu trúc nào lấy-ra-một-phần-tử, thêm, và kiểm tra thuộc đều `O(1)`?
   - **`set`** (lấy bất kỳ bằng `pop`);
   - **hàng đợi `deque`** (lấy đầu, thêm cuối) kèm **mảng cờ** để `check` và chặn trùng;
   - **bộ đếm `next` + ngăn xếp số đã trả** (không cần liệt kê trước mọi số!).
3. **Bất biến cần giữ:** mỗi số **thuộc đúng một** trạng thái (trống/đã cấp), và cấu trúc "trống" **không chứa trùng**. Mọi thao tác phải bảo toàn điều này; chú ý các thao tác lặp lại (`release` hai lần).
4. Đề cho phép `get` trả **bất kỳ** số trống: ta không bị buộc cấp số nhỏ nhất — chính sự tự do đó cho phép lời giải `O(1)`.

## 7. Các cách tiếp theo

### Cách 2 — Tập `set` các số còn trống

**Giải thích từng đoạn:**

- `self.available = set(range(maxNumbers))`: ban đầu mọi số đều trống.
- `get`: nếu `available` rỗng → `-1`; ngược lại `available.pop()` (lấy ra **và xóa** một số bất kỳ) rồi trả về.
- `check(number)`: `number in available`.
- `release(number)`: `available.add(number)` (thêm lần nữa vô hại vì `set` không có trùng).

In [10]:
class PhoneDirectorySet:
    def __init__(self, maxNumbers: int):
        self.available = set(range(maxNumbers))

    def get(self) -> int:
        if not self.available:
            return -1
        return self.available.pop()

    def check(self, number: int) -> bool:
        return number in self.available

    def release(self, number: int) -> None:
        self.available.add(number)

out = run_commands(PhoneDirectorySet, example_names, example_args)
print(out)

[None, 0, 1, True, 2, False, None, True]


Kết quả `get` của `set` có thể không phải `0,1,2` đúng thứ tự (Python không đảm bảo thứ tự), nhưng **vẫn đúng đề**. Ở mục 12 ta kiểm tra bằng **mô hình tham chiếu** thay vì so khớp từng số.

**Code nộp LeetCode (Cách 2):** chính là lớp trên với tên `PhoneDirectory`.

In [11]:
class PhoneDirectory:
    def __init__(self, maxNumbers: int):
        self.available = set(range(maxNumbers))

    def get(self) -> int:
        if not self.available:
            return -1
        return self.available.pop()

    def check(self, number: int) -> bool:
        return number in self.available

    def release(self, number: int) -> None:
        self.available.add(number)

d = PhoneDirectory(3)
print(d.get() in (0, 1, 2), d.check(2) in (True, False))

True True


**Độ phức tạp (Cách 2):** `get`, `check`, `release` đều `O(1)` trung bình; khởi tạo `O(maxNumbers)`; bộ nhớ `O(maxNumbers)`.

### Cách 3 — Hàng đợi `deque` + mảng cờ chống trùng

**Ý tưởng:** `queue` chứa các số trống theo thứ tự (FIFO); `is_free[i]` ghi cờ để `check` `O(1)` và để **`release` không thêm số đã trống** (tránh trùng trong hàng đợi).

**Giải thích từng đoạn:**

- `self.queue = deque(range(maxNumbers))`, `self.is_free = [True] * maxNumbers`.
- `get`: nếu `queue` rỗng → `-1`; ngược lại `x = queue.popleft()`, `is_free[x] = False`, trả `x`.
- `check`: `is_free[number]`.
- `release`: **chỉ khi `not is_free[number]`**: `is_free[number] = True` và `queue.append(number)`.

In [12]:
class PhoneDirectoryQueue:
    def __init__(self, maxNumbers: int):
        self.queue = deque(range(maxNumbers))
        self.is_free = [True] * maxNumbers

    def get(self) -> int:
        if not self.queue:
            return -1
        x = self.queue.popleft()
        self.is_free[x] = False
        return x

    def check(self, number: int) -> bool:
        return self.is_free[number]

    def release(self, number: int) -> None:
        if not self.is_free[number]:           # chặn: số đã trống thì không thêm lần nữa
            self.is_free[number] = True
            self.queue.append(number)

out = run_commands(PhoneDirectoryQueue, example_names, example_args)
print(out)
assert out == example_output

[None, 0, 1, True, 2, False, None, True]


**Độ phức tạp (Cách 3):** mọi thao tác `O(1)`; khởi tạo `O(maxNumbers)`; bộ nhớ `O(maxNumbers)`. Cấp số theo thứ tự `0,1,2,...` rồi các số được trả (công bằng kiểu FIFO).

### Cách 4 — Bộ đếm `next` + ngăn xếp số đã trả (khởi tạo lười)

**Ý tưởng khác:** **không liệt kê trước** mọi số. Nhớ một con số `next_free`: mọi số `>= next_free` **chưa từng được cấp** (nên đang trống). Số **đã từng cấp rồi được trả** đưa vào một danh sách `recycled`. Số đang cấp lưu trong `used`.

**Giải thích từng đoạn:**

- `self.next_free = 0`, `self.recycled = []`, `self.used = set()`, `self.max = maxNumbers`.
- `get`: nếu `recycled` có phần tử → `x = recycled.pop()`; elif `next_free < max` → `x = next_free; next_free += 1`; ngược lại `-1`. Cấp xong: `used.add(x)`.
- `check(number)`: `number not in used`.
- `release(number)`: **chỉ khi `number in used`**: `used.remove(number)` và `recycled.append(number)`.

Khởi tạo `O(1)` thay vì `O(maxNumbers)`; bộ nhớ `O(số đã cấp)`.

In [13]:
class PhoneDirectoryLazy:
    def __init__(self, maxNumbers: int):
        self.max = maxNumbers
        self.next_free = 0               # mọi số >= next_free chưa từng cấp
        self.recycled = []               # các số đã cấp rồi được trả (ngăn xếp)
        self.used = set()                # các số đang được cấp

    def get(self) -> int:
        if self.recycled:
            x = self.recycled.pop()
        elif self.next_free < self.max:
            x = self.next_free
            self.next_free += 1
        else:
            return -1
        self.used.add(x)
        return x

    def check(self, number: int) -> bool:
        return number not in self.used

    def release(self, number: int) -> None:
        if number in self.used:
            self.used.remove(number)
            self.recycled.append(number)

out = run_commands(PhoneDirectoryLazy, example_names, example_args)
print(out)
assert out == example_output

[None, 0, 1, True, 2, False, None, True]


**Độ phức tạp (Cách 4):** mọi thao tác `O(1)`; khởi tạo `O(1)`; bộ nhớ `O(số lượng số đã từng cấp)` ≤ `O(maxNumbers)`. Cấp lại số vừa trả gần nhất (kiểu LIFO).

### Cách 5 — Bitmask bằng một số nguyên lớn

**Ý tưởng:** cả "bản đồ trống/đã cấp" nằm trong **một số nguyên** `mask`; bit `i = 1` nghĩa là số `i` **trống**. Mục 3.5 đã giới thiệu các phép bit.

**Giải thích từng đoạn:**

- `self.mask = (1 << maxNumbers) - 1` (mọi bit từ 0 đến `n-1` bằng 1).
- `get`: nếu `mask == 0` → `-1`; ngược lại `low = mask & -mask` (bit thấp nhất), `mask ^= low` (tắt bit), trả `low.bit_length() - 1`.
- `check`: `(mask >> number) & 1 == 1`.
- `release`: `mask |= 1 << number` (bật bit; bật lần nữa không đổi gì — tự chống trùng).

In [14]:
class PhoneDirectoryBits:
    def __init__(self, maxNumbers: int):
        self.mask = (1 << maxNumbers) - 1

    def get(self) -> int:
        if self.mask == 0:
            return -1
        low = self.mask & -self.mask             # bit 1 thấp nhất = số trống nhỏ nhất
        self.mask ^= low
        return low.bit_length() - 1

    def check(self, number: int) -> bool:
        return (self.mask >> number) & 1 == 1

    def release(self, number: int) -> None:
        self.mask |= 1 << number

out = run_commands(PhoneDirectoryBits, example_names, example_args)
print(out)
assert out == example_output

[None, 0, 1, True, 2, False, None, True]


**Độ phức tạp (Cách 5):** mỗi thao tác `O(maxNumbers / 64)` (số nguyên lớn ~ `n/64` từ máy): với `n = 10^4` chỉ ~`157` từ — rất nhanh trong thực tế, nhưng về lý thuyết không phải `O(1)`. Bộ nhớ `O(n)` bit (≈ `n/8` byte, nhỏ nhất trong các cách). Luôn cấp số **nhỏ nhất**.

## 8. TRACE từng bước

Ta trace **Cách 4 (lazy)** với ví dụ đề (và thêm vài thao tác góc cạnh): in sau mỗi lệnh các biến `next_free`, `recycled`, `used`.

| lệnh | kết quả | `next_free` | `recycled` | `used` |
|---|---|---|---|---|
| `PhoneDirectory(3)` | null | 0 | [] | {} |
| `get()` | 0 | 1 | [] | {0} |
| `get()` | 1 | 2 | [] | {0,1} |
| `check(2)` | true (2 ∉ used) | 2 | [] | {0,1} |
| `get()` | 2 | 3 | [] | {0,1,2} |
| `check(2)` | false | 3 | [] | {0,1,2} |
| `release(2)` | null | 3 | [2] | {0,1} |
| `check(2)` | true | 3 | [2] | {0,1} |

In [15]:
def trace_lazy(names, args):
    obj = None
    print(f"{'lệnh':<22} {'kết quả':<8} {'next_free':<10} {'recycled':<10} used")
    for name, a in zip(names, args):
        if name == "PhoneDirectory":
            obj = PhoneDirectoryLazy(*a)
            kq = None
        else:
            kq = getattr(obj, name)(*a)
        lenh = f"{name}({', '.join(map(str, a))})"
        print(f"{lenh:<22} {str(kq):<8} {obj.next_free:<10} {str(obj.recycled):<10} {sorted(obj.used)}")

trace_lazy(example_names, example_args)

lệnh                   kết quả  next_free  recycled   used
PhoneDirectory(3)      None     0          []         []
get()                  0        1          []         [0]
get()                  1        2          []         [0, 1]
check(2)               True     2          []         [0, 1]
get()                  2        3          []         [0, 1, 2]
check(2)               False    3          []         [0, 1, 2]
release(2)             None     3          [2]        [0, 1]
check(2)               True     3          [2]        [0, 1]


Thêm một kịch bản **góc cạnh**: kho hết, `release` hai lần, `release` số chưa từng cấp.

In [16]:
names = ["PhoneDirectory", "get", "get", "get", "release", "release", "release", "check", "get", "get", "get"]
args = [[2], [], [], [], [0], [0], [1], [0], [], [], []]
trace_lazy(names, args)
print("Tất cả Cách 4:", run_commands(PhoneDirectoryLazy, names, args))

lệnh                   kết quả  next_free  recycled   used
PhoneDirectory(2)      None     0          []         []
get()                  0        1          []         [0]
get()                  1        2          []         [0, 1]
get()                  -1       2          []         [0, 1]
release(0)             None     2          [0]        [1]
release(0)             None     2          [0]        [1]
release(1)             None     2          [0, 1]     []
check(0)               True     2          [0, 1]     []
get()                  1        2          [0]        [1]
get()                  0        2          []         [0, 1]
get()                  -1       2          []         [0, 1]
Tất cả Cách 4: [None, 0, 1, -1, None, None, None, True, 1, 0, -1]


Quan sát: `get` thứ ba (kho hết) → `-1`; `release(0)` hai lần chỉ thêm `0` vào `recycled` **một lần** nhờ kiểm tra `if number in used`; `release(1)` khi `1` đang được cấp (bước `get` thứ hai) là hợp lệ.

### Trace Cách 3 (hàng đợi): hàng đợi và cờ qua từng lệnh

In [17]:
def trace_queue(names, args):
    obj = None
    print(f"{'lệnh':<16} {'kết quả':<8} {'queue':<14} is_free")
    for name, a in zip(names, args):
        if name == "PhoneDirectory":
            obj = PhoneDirectoryQueue(*a)
            kq = None
        else:
            kq = getattr(obj, name)(*a)
        lenh = f"{name}({', '.join(map(str, a))})"
        print(f"{lenh:<16} {str(kq):<8} {str(list(obj.queue)):<14} {obj.is_free}")

trace_queue(example_names, example_args)

lệnh             kết quả  queue          is_free
PhoneDirectory(3) None     [0, 1, 2]      [True, True, True]
get()            0        [1, 2]         [False, True, True]
get()            1        [2]            [False, False, True]
check(2)         True     [2]            [False, False, True]
get()            2        []             [False, False, False]
check(2)         False    []             [False, False, False]
release(2)       None     [2]            [False, False, True]
check(2)         True     [2]            [False, False, True]


## 9. Code nộp LeetCode

Ta nộp **Cách 4 (lazy)** hoặc **Cách 2 (set)**. Cách 2 ngắn nhất; Cách 4 tiết kiệm khởi tạo. Dưới đây là **Cách 2** với tên lớp đúng `PhoneDirectory` (đã định nghĩa ở mục 7, bản "cuối cùng" ghi lại đây cho đầy đủ và kèm bộ chạy ví dụ).

In [18]:
class PhoneDirectory:
    def __init__(self, maxNumbers: int):
        # available: tập các số hiện đang TRỐNG (chưa gán cho ai)
        self.available = set(range(maxNumbers))

    def get(self) -> int:
        if not self.available:
            return -1                      # hết số
        return self.available.pop()        # lấy ra và xóa một số trống bất kỳ

    def check(self, number: int) -> bool:
        return number in self.available

    def release(self, number: int) -> None:
        self.available.add(number)         # set không có trùng -> release lặp lại vô hại


# Your PhoneDirectory object will be instantiated and called as such:
# obj = PhoneDirectory(maxNumbers)
# param_1 = obj.get()
# param_2 = obj.check(number)
# obj.release(number)

out = run_commands(PhoneDirectory, example_names, example_args)
print(out)

[None, 0, 1, True, 2, False, None, True]


Vì `get` có thể trả số bất kỳ, ta không dùng `assert out == example_output` mà kiểm tra bằng **mô hình tham chiếu** (mục 12). Phần dưới đây là một bộ kiểm tra tính hợp lệ nhanh cho ví dụ:

In [19]:
def kiem_tra_hop_le(cls, names, args):
    """Chạy chuỗi lệnh trên cls và kiểm tra từng kết quả bằng mô hình tham chiếu (set các số trống).
    Trả danh sách kết quả; ném AssertionError nếu có lệnh sai."""
    obj, model, out, maxn = None, set(), [], 0
    for i, (name, a) in enumerate(zip(names, args)):
        if name == "PhoneDirectory":
            obj = cls(*a)
            maxn = a[0]
            model = set(range(maxn))
            out.append(None)
        elif name == "get":
            r = obj.get()
            if model:
                assert r in model, f"lệnh {i}: get trả {r} nhưng số này không còn trống (model={sorted(model)[:10]}...)"
                model.remove(r)
            else:
                assert r == -1, f"lệnh {i}: kho hết nhưng get trả {r} thay vì -1"
            out.append(r)
        elif name == "check":
            r = obj.check(*a)
            assert r is (a[0] in model), f"lệnh {i}: check({a[0]}) trả {r}, mong đợi {a[0] in model}"
            out.append(r)
        elif name == "release":
            r = obj.release(*a)
            assert r is None, f"lệnh {i}: release phải trả None, nhận {r}"
            model.add(a[0])
            out.append(None)
        else:
            raise ValueError(name)
    return out

for cls in (PhoneDirectory, PhoneDirectoryScan, PhoneDirectorySet, PhoneDirectoryQueue, PhoneDirectoryLazy, PhoneDirectoryBits):
    print(f"{cls.__name__:<22}", kiem_tra_hop_le(cls, example_names, example_args))

PhoneDirectory         [None, 0, 1, True, 2, False, None, True]
PhoneDirectoryScan     [None, 0, 1, True, 2, False, None, True]
PhoneDirectorySet      [None, 0, 1, True, 2, False, None, True]
PhoneDirectoryQueue    [None, 0, 1, True, 2, False, None, True]
PhoneDirectoryLazy     [None, 0, 1, True, 2, False, None, True]
PhoneDirectoryBits     [None, 0, 1, True, 2, False, None, True]


## 10. Vì sao thuật toán đúng

Gọi `F` là tập các số **đang trống** theo định nghĩa đề. Ta chứng minh mỗi cách duy trì **bất biến** `S = F`, trong đó `S` là tập mà cấu trúc dữ liệu "nghĩ" là đang trống.

**Cách 2 (set):**
1. Ban đầu `S = {0..n-1} = F`.
2. `get`: nếu `S` rỗng thì `F` rỗng → `-1` đúng; ngược lại `pop` lấy `x ∈ S = F` rồi bỏ `x` khỏi `S` (số vừa cấp không còn trống) — `S = F` vẫn đúng và `x` hợp lệ.
3. `check(i)`: `i ∈ S ⇔ i ∈ F`.
4. `release(i)`: thêm `i` vào `S`; đề muốn `i` trống nên `F` cũng thêm `i`. Nếu `i` đã trống thì `set.add` không đổi gì (idempotent).

**Cách 3 (hàng đợi + cờ):** bất biến: `queue` chứa **đúng một lần** mỗi số có `is_free = True` (và không chứa số `is_free = False`). `get` lấy đầu hàng, đặt cờ `False` — giữ bất biến. `release` chỉ thêm khi cờ là `False` rồi đặt `True` — số được thêm **đúng một lần**; nếu cờ đã `True` thì không thêm → không trùng. (Bỏ kiểm tra cờ sẽ vi phạm bất biến — xem lỗi 1 ở mục 13.)

**Cách 4 (lazy):** tập trống `F = {x : x ≥ next_free} ∪ recycled`. Hai phần **rời nhau** (số trong `recycled` đều `< next_free` vì đã từng cấp). `get` lấy từ một trong hai phần rồi đưa vào `used`; `release` chỉ chuyển số `x ∈ used` sang `recycled`. Số chưa từng cấp mà bị `release` (ngoài `used`) được bỏ qua — nó vốn đã trống và nằm trong phần `≥ next_free`. `check(i) = i ∉ used` đúng vì `F` là phần bù của `used` trong `[0, n)`.

**Cách 5 (bitmask):** bit `i` của `mask` = `[i ∈ F]`. `get` tắt đúng một bit đang bật; `release` bật bit `i` (bật lại bit đã bật không đổi); `check` đọc bit. Rõ ràng.

**Cách 1 (quét):** `is_free[i] = [i ∈ F]`; `get` đổi một cờ `True` thành `False`.

Ta cũng kiểm chứng bằng **mô hình tham chiếu** trên hàng nghìn chuỗi lệnh ngẫu nhiên (mục 12).

## 11. Phân tích Time / Space Complexity

| Cách | `get` | `check` | `release` | Khởi tạo | Space |
|---|---|---|---|---|---|
| 1. Mảng cờ + quét | `O(n)` | `O(1)` | `O(1)` | `O(n)` | `O(n)` |
| 2. `set` | `O(1)` | `O(1)` | `O(1)` | `O(n)` | `O(n)` |
| 3. `deque` + cờ | `O(1)` | `O(1)` | `O(1)` | `O(n)` | `O(n)` |
| 4. Bộ đếm + ngăn xếp (lười) | `O(1)` | `O(1)` | `O(1)` | `O(1)` | `O(số đã cấp)` |
| 5. Bitmask | `O(n/64)` | `O(n/64)` | `O(n/64)` | `O(n/64)` | `O(n)` bit |

(`n = maxNumbers`.) Với `n = 10^4` và `2·10^4` lệnh, các cách 2–5 chạy trong chưa đầy vài chục mili-giây; chỉ cách 1 có thể chậm.

In [20]:
def tao_lenh(rng, maxn, so_lenh, ti_le=(1, 1, 1)):
    """Sinh chuỗi lệnh ngẫu nhiên hợp lệ theo ràng buộc đề: number trong [0, maxn)."""
    names, args = ["PhoneDirectory"], [[maxn]]
    for _ in range(so_lenh):
        loai = rng.choices(["get", "check", "release"], weights=ti_le)[0]
        names.append(loai)
        args.append([] if loai == "get" else [rng.randrange(maxn)])
    return names, args

rng = random.Random(379)
names, args = tao_lenh(rng, 10000, 20000, ti_le=(3, 1, 2))
for cls in (PhoneDirectoryScan, PhoneDirectorySet, PhoneDirectoryQueue, PhoneDirectoryLazy, PhoneDirectoryBits):
    t = time.perf_counter()
    run_commands(cls, names, args)
    print(f"{cls.__name__:<22}: {1000 * (time.perf_counter() - t):8.2f} ms  (n = 10^4, 2*10^4 lệnh)")

PhoneDirectoryScan    :   655.65 ms  (n = 10^4, 2*10^4 lệnh)
PhoneDirectorySet     :     4.53 ms  (n = 10^4, 2*10^4 lệnh)
PhoneDirectoryQueue   :     4.04 ms  (n = 10^4, 2*10^4 lệnh)
PhoneDirectoryLazy    :     4.45 ms  (n = 10^4, 2*10^4 lệnh)
PhoneDirectoryBits    :    14.00 ms  (n = 10^4, 2*10^4 lệnh)


## 12. Bộ test

Vì `get` được phép trả **bất kỳ** số trống, ta **không** so từng đầu ra với một danh sách cứng. Thay vào đó dùng `kiem_tra_hop_le` (mục 9): mô hình tham chiếu giữ **tập số đang trống** và kiểm tra: `get` trả số **đang trống** (hoặc `-1` đúng lúc kho hết), `check` đúng trạng thái, `release` trả `None`.

### 12.1. Ví dụ đề và các trường hợp biên

In [21]:
tat_ca = (PhoneDirectory, PhoneDirectoryScan, PhoneDirectorySet, PhoneDirectoryQueue, PhoneDirectoryLazy, PhoneDirectoryBits)

# 1) ví dụ đề
for cls in tat_ca:
    kiem_tra_hop_le(cls, example_names, example_args)

# 2) maxNumbers = 1
n1 = ["PhoneDirectory", "get", "get", "check", "release", "check", "get", "get"]
a1 = [[1], [], [], [0], [0], [0], [], []]
for cls in tat_ca:
    kq = kiem_tra_hop_le(cls, n1, a1)
    assert kq == [None, 0, -1, False, None, True, 0, -1], (cls.__name__, kq)

# 3) release số đã trống (không làm gì), rồi cạn kho: phải cấp ĐÚNG 3 số khác nhau rồi -1
n2 = ["PhoneDirectory", "release", "release", "get", "get", "get", "get"]
a2 = [[3], [0], [0], [], [], [], []]
for cls in tat_ca:
    kq = kiem_tra_hop_le(cls, n2, a2)
    cap = kq[3:6]
    assert sorted(cap) == [0, 1, 2] and kq[6] == -1, (cls.__name__, kq)

# 4) cạn kho hoàn toàn rồi trả hết rồi cấp lại hết
n3 = ["PhoneDirectory"] + ["get"] * 5 + ["release"] * 5 + ["get"] * 6
a3 = [[5]] + [[]] * 5 + [[i] for i in range(5)] + [[]] * 6
for cls in tat_ca:
    kq = kiem_tra_hop_le(cls, n3, a3)
    assert sorted(kq[1:6]) == [0, 1, 2, 3, 4]
    assert sorted(kq[11:16]) == [0, 1, 2, 3, 4] and kq[16] == -1
print("Đã qua: ví dụ đề, maxNumbers=1, release số đã trống, cạn kho rồi cấp lại -", len(tat_ca), "cách")

Đã qua: ví dụ đề, maxNumbers=1, release số đã trống, cạn kho rồi cấp lại - 6 cách


### 12.2. Các cách làm sai (phản ví dụ) — bộ kiểm tra phải bắt được

Ta viết sẵn vài cài đặt **sai** và chứng minh `kiem_tra_hop_le` phát hiện ra:

1. **Hàng đợi không chặn trùng** (bỏ kiểm tra cờ khi `release`).
2. **`get` không đánh dấu số vừa cấp** (dùng `next(iter(set))` mà quên xóa).
3. **Không trả `-1` khi hết** (ném lỗi hoặc trả `None`).

In [22]:
class SaiQueueTrung(PhoneDirectoryQueue):
    def release(self, number):
        self.is_free[number] = True
        self.queue.append(number)               # SAI: không kiểm tra cờ -> trùng trong hàng đợi

class SaiKhongXoa:
    def __init__(self, maxNumbers):
        self.available = set(range(maxNumbers))
    def get(self):
        if not self.available:
            return -1
        return next(iter(self.available))       # SAI: lấy mà không xóa khỏi tập
    def check(self, number):
        return number in self.available
    def release(self, number):
        self.available.add(number)

class SaiKhongTraMot(PhoneDirectorySet):
    def get(self):
        if not self.available:
            return None                         # SAI: phải trả -1
        return self.available.pop()

def bat_loi(cls, names, args):
    try:
        kiem_tra_hop_le(cls, names, args)
        return "KHÔNG bắt được lỗi!"
    except AssertionError as e:
        return "bắt được: " + str(e)[:90]

# Kịch bản 1: release(0) khi 0 đã trống -> queue có hai bản sao của 0 -> sau 3 lần get (hết 3 số) lần get thứ 4 vẫn ra 0
k1n = ["PhoneDirectory", "release", "get", "get", "get", "get"]
k1a = [[3], [0], [], [], [], []]
print("1)", bat_loi(SaiQueueTrung, k1n, k1a))
# Kịch bản 2: get xong check phải false
k2n = ["PhoneDirectory", "get", "get"]
k2a = [[1], [], []]
print("2)", bat_loi(SaiKhongXoa, k2n, k2a))
# Kịch bản 3: get khi hết
k3n = ["PhoneDirectory", "get", "get"]
k3a = [[1], [], []]
print("3)", bat_loi(SaiKhongTraMot, k3n, k3a))
assert all("bắt được" in bat_loi(c, n, a) for c, n, a in [(SaiQueueTrung, k1n, k1a), (SaiKhongXoa, k2n, k2a), (SaiKhongTraMot, k3n, k3a)])

1) bắt được: lệnh 5: kho hết nhưng get trả 0 thay vì -1
2) bắt được: lệnh 2: kho hết nhưng get trả 0 thay vì -1
3) bắt được: lệnh 2: kho hết nhưng get trả None thay vì -1


### 12.3. Đối chiếu ngẫu nhiên (có seed) — mô hình tham chiếu cho cả 6 cài đặt

Sinh chuỗi lệnh ngẫu nhiên đúng ràng buộc (`number` trong `[0, maxNumbers)`), với nhiều tỉ lệ `get/check/release` khác nhau (để có lúc kho cạn, lúc đầy) rồi kiểm tra từng cài đặt bằng mô hình.

In [23]:
rng = random.Random(3790)
dem = 0
for _ in range(600):
    maxn = rng.choice([1, 2, 3, 5, 8, 20, 60])
    ti_le = rng.choice([(1, 1, 1), (4, 1, 1), (1, 1, 4), (5, 0, 1), (2, 3, 2)])
    names, args = tao_lenh(rng, maxn, rng.randint(1, 80), ti_le)
    for cls in tat_ca:
        kiem_tra_hop_le(cls, names, args)
    dem += 1
print("6 cài đặt đúng theo mô hình trên", dem, "chuỗi lệnh ngẫu nhiên nhỏ")

6 cài đặt đúng theo mô hình trên 600 chuỗi lệnh ngẫu nhiên nhỏ


In [24]:
# Cỡ đúng ràng buộc đề: maxNumbers tới 10^4 và 2*10^4 lệnh. (Bỏ Scan vì chậm O(n) mỗi get.)
rng = random.Random(37900)
nhanh = (PhoneDirectory, PhoneDirectorySet, PhoneDirectoryQueue, PhoneDirectoryLazy, PhoneDirectoryBits)
for lan in range(6):
    maxn = rng.choice([1, 100, 5000, 10000])
    ti_le = rng.choice([(1, 1, 1), (4, 1, 1), (1, 1, 4)])
    names, args = tao_lenh(rng, maxn, 20000, ti_le)
    for cls in nhanh:
        kiem_tra_hop_le(cls, names, args)
print("5 cài đặt nhanh đúng theo mô hình trên 6 chuỗi 2*10^4 lệnh, maxNumbers tới 10^4")

5 cài đặt nhanh đúng theo mô hình trên 6 chuỗi 2*10^4 lệnh, maxNumbers tới 10^4


In [25]:
# Cài đặt xác định (Scan, Queue, Lazy, Bits) phải cho kết quả GIỐNG HỆT nhau khi cùng chính sách cấp số:
# Scan và Bits luôn cấp số nhỏ nhất -> hai cài đặt này phải khớp nhau từng lệnh.
rng = random.Random(379000)
for _ in range(300):
    maxn = rng.choice([1, 2, 4, 9, 30])
    names, args = tao_lenh(rng, maxn, rng.randint(1, 60), rng.choice([(1, 1, 1), (3, 1, 2)]))
    assert run_commands(PhoneDirectoryScan, names, args) == run_commands(PhoneDirectoryBits, names, args)
print("Scan và Bits (cùng cấp số nhỏ nhất) khớp từng lệnh trên 300 chuỗi")

Scan và Bits (cùng cấp số nhỏ nhất) khớp từng lệnh trên 300 chuỗi


## 13. Các lỗi người mới hay gặp

### Lỗi 1: `release` số đã trống làm nhân đôi trong hàng đợi

Với cách hàng đợi, nếu `release` luôn `append` mà không kiểm tra cờ, số trống bị nằm **hai lần** trong hàng đợi và có thể bị cấp **hai lần** cho hai người.

In [26]:
d = SaiQueueTrung(2)
d.release(0)                                 # 0 đang trống mà vẫn bị thêm lần nữa
print("hàng đợi:", list(d.queue))            # [0, 1, 0]
print("get:", d.get(), d.get(), d.get())     # 0 1 0 -> số 0 bị cấp hai lần!

hàng đợi: [0, 1, 0]
get: 0 1 0


Vì sao sai: bất biến "mỗi số trống xuất hiện đúng một lần" bị phá vỡ. **Đúng:** chỉ thêm khi cờ cho biết số đó **đang được cấp** (`if not is_free[number]`), như Cách 3.

### Lỗi 2: `get` quên đánh dấu số đã cấp

Dùng `next(iter(set))` hoặc đọc phần tử mà không `remove`: số vừa cấp vẫn "trống", `check` trả `True` và `get` tiếp theo cấp lại **đúng số đó**.

In [27]:
d = SaiKhongXoa(3)
print(d.get(), d.get(), d.check(0))          # 0 0 True  (số 0 bị cấp lặp, và vẫn "trống")

0 0 True


**Đúng:** dùng `set.pop()` (lấy **và xóa**) hoặc `remove` ngay sau khi lấy.

### Lỗi 3: Sửa `set` khi đang duyệt nó

Ví dụ viết `get` kiểu: `for x in self.available: self.available.remove(x); return x`. Với `return` ngay thì tạm ổn, nhưng nếu quên `return` hoặc `break` thì Python báo `RuntimeError: Set changed size during iteration`.

In [28]:
s = {1, 2, 3}
try:
    for x in s:
        s.remove(x)                          # SAI: sửa tập trong khi duyệt
except RuntimeError as e:
    print("RuntimeError:", e)

RuntimeError: Set changed size during iteration


**Đúng:** lấy phần tử bằng `pop()` (không cần vòng lặp).

### Lỗi 4: Dùng `list.pop(0)` làm hàng đợi

`list.pop(0)` phải dịch mọi phần tử còn lại → `O(n)` mỗi lần. Với `n = 10^4` và nhiều lệnh, thời gian tăng đáng kể so với `deque.popleft()` `O(1)`.

In [29]:
def dem_thoi_gian_pop(lam_hang_doi, n):
    h = lam_hang_doi(range(n))
    t = time.perf_counter()
    if isinstance(h, list):
        while h:
            h.pop(0)
    else:
        while h:
            h.popleft()
    return 1000 * (time.perf_counter() - t)

for n in (20000, 40000, 80000):
    print(f"n={n:>6}: list.pop(0) = {dem_thoi_gian_pop(list, n):8.2f} ms | deque.popleft() = {dem_thoi_gian_pop(deque, n):6.2f} ms")

n= 20000: list.pop(0) =    30.03 ms | deque.popleft() =   0.51 ms
n= 40000: list.pop(0) =   123.23 ms | deque.popleft() =   1.06 ms


n= 80000: list.pop(0) =   489.72 ms | deque.popleft() =   1.98 ms


**Đúng:** dùng `collections.deque` cho hàng đợi.

### Lỗi 5: Không trả `-1` khi hết số (hoặc `check` / `release` ném lỗi)

`set.pop()` trên tập rỗng ném `KeyError`; nhớ kiểm tra rỗng trước.

In [30]:
s = set()
try:
    s.pop()
except KeyError as e:
    print("KeyError khi pop tập rỗng:", repr(e))

d = PhoneDirectory(1)
print(d.get(), d.get())                      # số nào đó rồi -1 (đã xử lý)

KeyError khi pop tập rỗng: KeyError('pop from an empty set')
0 -1


### Lỗi 6: Test sai vì giả định `get` trả đúng số nhỏ nhất

Đề **không** yêu cầu thứ tự. Một test chỉ pass với `[0, 1, 2]` có thể làm bạn nghĩ lời giải `set` sai trong khi nó đúng. Luôn kiểm tra bằng **tính chất** (số trả về đang trống, `-1` đúng lúc hết...) như `kiem_tra_hop_le`.

## 14. Mẹo & nhận dạng pattern

- **Từ khóa:** "Design", "initialize ... get ... release/put/remove ... check/contains" → bài **quản lý tập tài nguyên** (ID, số điện thoại, ghế ngồi, cổng mạng...). Câu hỏi đầu tiên: *cần cấu trúc nào cho mỗi thao tác đạt `O(1)`?*
- **Bộ ba thường dùng:** `set` (thuộc/thêm/xóa `O(1)`), `deque`/ngăn xếp (lấy một phần tử nhanh), mảng cờ (truy cập theo chỉ số).
- **Khởi tạo lười (lazy)** — con trỏ `next` + danh sách tái chế — là mẹo hay: tránh tạo sẵn `n` phần tử. Dùng ở *Smallest Number in Infinite Set* (2336), *Seat Reservation Manager* (1845).
- **Luôn tự hỏi về tính idempotent:** "gọi lại thao tác hai lần có hỏng không?" (`release` số đã trống).
- **Khi đề cho `get` trả "bất kỳ":** tận dụng — chọn cấu trúc cho `O(1)`. Khi đề đòi "nhỏ nhất": dùng **min-heap** (hoặc bitmask / balanced tree).
- **Cách test bài Design:** viết **bộ chạy chuỗi lệnh** (như `run_commands`) + **mô hình tham chiếu** thật đơn giản (`set` Python) rồi so với cài đặt của bạn trên nhiều chuỗi ngẫu nhiên có seed.
- **Họ bài:** *Design HashSet* (705), *Design HashMap* (706), *Insert Delete GetRandom O(1)* (380), *LRU Cache* (146), *Design Twitter* (355), *Design Hit Counter* (362).

## 15. Biến thể

### 15.1. `get` phải trả số **nhỏ nhất** còn trống

Dùng **min-heap** các số trống + tập cờ để chống trùng (hoặc bitmask / Cách 1 nhưng nhanh hơn). `get`: `heappop`; `release`: chỉ `heappush` khi số đang được cấp.

In [31]:
import heapq

class PhoneDirectorySmallest:
    def __init__(self, maxNumbers: int):
        self.heap = list(range(maxNumbers))      # list tăng dần đã là min-heap hợp lệ
        self.is_free = [True] * maxNumbers

    def get(self) -> int:
        if not self.heap:
            return -1
        x = heapq.heappop(self.heap)
        self.is_free[x] = False
        return x

    def check(self, number: int) -> bool:
        return self.is_free[number]

    def release(self, number: int) -> None:
        if not self.is_free[number]:
            self.is_free[number] = True
            heapq.heappush(self.heap, number)

# Mô hình "nhỏ nhất": so từng lệnh với Scan (cũng cấp số nhỏ nhất)
rng = random.Random(1)
for _ in range(300):
    maxn = rng.choice([1, 3, 8, 25])
    names, args = tao_lenh(rng, maxn, rng.randint(1, 80), rng.choice([(1, 1, 1), (3, 1, 2)]))
    assert run_commands(PhoneDirectorySmallest, names, args) == run_commands(PhoneDirectoryScan, names, args)
print("Min-heap luôn cấp số nhỏ nhất, khớp với Scan; get O(log n)")

Min-heap luôn cấp số nhỏ nhất, khớp với Scan; get O(log n)


### 15.2. `release` số chưa từng cấp / ngoài phạm vi

Đề đảm bảo `0 <= number < maxNumbers`. Nếu cho phép số ngoài phạm vi, cần kiểm tra `0 <= number < max` trước (đa số cài đặt sẽ ném `IndexError` hoặc thêm số "ma" vào kho).

### 15.3. Các biến thể khác

- **Seat Reservation Manager (1845):** `reserve()` trả ghế **nhỏ nhất**, `unreserve(seat)` — chính là 15.1.
- **Smallest Number in Infinite Set (2336):** kho **vô hạn** `1,2,3,...` → khởi tạo lười bắt buộc (bài tập 5).
- **Thread-safe:** thêm khóa (`threading.Lock`) cho mỗi thao tác.
- **Có hạn dùng (expire):** thêm thời gian, dùng heap theo thời điểm hết hạn.
- **Thêm API `used_count()`:** giữ biến đếm thay vì tính lại.

## 16. So sánh các cách làm

| Cách | Ý tưởng | `get` | Khởi tạo | Space | Khi nào dùng |
|---|---|---|---|---|---|
| 1. Mảng cờ + quét | Quét tìm ô trống | `O(n)` | `O(n)` | `O(n)` | Chỉ để hiểu đề / `n` rất nhỏ |
| 2. `set` | Giữ tập số trống, `pop` | `O(1)` | `O(n)` | `O(n)` | **Ngắn gọn nhất**, nộp bài |
| 3. `deque` + cờ | FIFO + chặn trùng | `O(1)` | `O(n)` | `O(n)` | Muốn thứ tự công bằng, tránh phụ thuộc thứ tự `set` |
| 4. Bộ đếm + tái chế | Khởi tạo lười | `O(1)` | `O(1)` | `O(đã cấp)` | Kho rất lớn/vô hạn; tiết kiệm bộ nhớ |
| 5. Bitmask | Một số nguyên làm bản đồ bit | `O(n/64)` | `O(n/64)` | `O(n)` bit | Muốn số **nhỏ nhất**, bộ nhớ nhỏ nhất |

In [32]:
rng = random.Random(16)
names, args = tao_lenh(rng, 10000, 20000, (3, 1, 2))
print(f"{'cài đặt':<22} {'thời gian (ms)':>14}")
for cls in (PhoneDirectorySet, PhoneDirectoryQueue, PhoneDirectoryLazy, PhoneDirectoryBits, PhoneDirectorySmallest):
    t = time.perf_counter()
    run_commands(cls, names, args)
    print(f"{cls.__name__:<22} {1000 * (time.perf_counter() - t):>14.2f}")

cài đặt                thời gian (ms)
PhoneDirectorySet                4.37
PhoneDirectoryQueue              3.84
PhoneDirectoryLazy               4.72
PhoneDirectoryBits              13.22
PhoneDirectorySmallest           5.59


## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code. Với `PhoneDirectoryQueue(3)`, chạy chuỗi `get, get, release(0), get, get, get`. Hãy đoán danh sách kết quả của **6** lệnh sau constructor. (Gợi ý: nhớ thứ tự FIFO của hàng đợi và chuyện `release(0)` khi `0` đang được cấp.)

**Bài 2 (tính tay).** Với `PhoneDirectoryLazy(4)`: `get, get, get, release(1), release(1), get, get, get, get`. Lập bảng `next_free / recycled / used` sau mỗi lệnh như mục 8 và nêu kết quả mỗi `get`.

**Bài 3 (điền TODO).** Hoàn thành lớp `PhoneDirectoryTodo` theo Cách 2 (dùng `set`).

In [33]:
class PhoneDirectoryTodo:
    def __init__(self, maxNumbers: int):
        # TODO 1: self.available = tập các số 0..maxNumbers-1
        self.available = None

    def get(self) -> int:
        # TODO 2: nếu tập rỗng trả -1; ngược lại lấy ra (pop) và trả một số
        return None

    def check(self, number: int) -> bool:
        # TODO 3: số có đang trống không
        return None

    def release(self, number: int) -> None:
        # TODO 4: thêm số vào tập trống
        return None

# KHÔNG tạo đối tượng PhoneDirectoryTodo ở đây để "Run All" không lỗi (chưa điền xong).

**Bài 4 (tự viết lại).** Viết `PhoneDirectoryBits2` bằng **bitmask** (Cách 5) **mà không nhìn lại** mục 7. Kiểm tra bằng `kiem_tra_hop_le` và so với `PhoneDirectoryScan` (cùng cấp số nhỏ nhất).

**Bài 5 (biến thể / mini challenge).** **Smallest Number in Infinite Set (2336)**: lớp `SmallestInfiniteSet` quản lý tập vô hạn `{1, 2, 3, ...}`:

- `popSmallest()`: xóa và trả số nhỏ nhất trong tập;
- `addBack(num)`: thêm `num` trở lại **nếu** hiện không có trong tập.

Kiểm tra: `popSmallest()` ba lần → `1, 2, 3`; `addBack(2)`; `popSmallest()` → `2`; `popSmallest()` → `4`; `addBack(10)` (10 chưa bị pop nên **không** thêm). Gợi ý: kết hợp **bộ đếm `cur`** (mọi số `>= cur` còn trong tập) và **min-heap + set** các số đã pop rồi được thêm lại.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: hàng đợi ban đầu `[0,1,2]`; `get,get` → `0,1`, hàng đợi `[2]`; `release(0)` → `0` đang được cấp nên được thêm cuối → `[2,0]`; ba `get` còn lại → `2, 0, -1`. Đáp án 6 lệnh sau constructor: `[0, 1, None, 2, 0, -1]`. Bài 3: `set(range(maxNumbers))`, nhớ `if not self.available`.
- **Mức 2 (vừa):** Bài 2: sau 3 `get` đầu: `next_free = 3`, `used = {0,1,2}`, kết quả `0,1,2`. `release(1)` → `recycled = [1]`; `release(1)` lần hai: `1` không còn trong `used` → bỏ qua. 4 `get` cuối: `1, 3, -1, -1`. Bài 4: `mask & -mask` rồi `.bit_length() - 1`.
- **Mức 3 (sâu):** Bài 5: `popSmallest`: nếu heap có phần tử thì `heappop` và bỏ khỏi `set`; ngược lại trả `cur` rồi `cur += 1`. `addBack(num)`: chỉ khi `num < cur` và `num` chưa có trong set mới `heappush`.

## 19. Đáp án tham khảo cho bài tập

In [34]:
# Bài 1: PhoneDirectoryQueue(3): get, get, release(0), get, get, get
kq1 = run_commands(PhoneDirectoryQueue, ["PhoneDirectory", "get", "get", "release", "get", "get", "get"],
                   [[3], [], [], [0], [], [], []])
print("Bài 1:", kq1)
assert kq1 == [None, 0, 1, None, 2, 0, -1]

# Bài 2: PhoneDirectoryLazy(4)
n2 = ["PhoneDirectory", "get", "get", "get", "release", "release", "get", "get", "get", "get"]
a2 = [[4], [], [], [], [1], [1], [], [], [], []]
kq2 = run_commands(PhoneDirectoryLazy, n2, a2)
print("Bài 2:", kq2)
assert kq2 == [None, 0, 1, 2, None, None, 1, 3, -1, -1]
trace_lazy(n2, a2)

# Bài 3
class PhoneDirectoryTodoAns:
    def __init__(self, maxNumbers: int):
        self.available = set(range(maxNumbers))
    def get(self) -> int:
        if not self.available:
            return -1
        return self.available.pop()
    def check(self, number: int) -> bool:
        return number in self.available
    def release(self, number: int) -> None:
        self.available.add(number)

rng = random.Random(3)
for _ in range(100):
    maxn = rng.choice([1, 2, 5, 12])
    names, args = tao_lenh(rng, maxn, rng.randint(1, 60))
    kiem_tra_hop_le(PhoneDirectoryTodoAns, names, args)
print("Bài 3 OK")

# Bài 4
class PhoneDirectoryBits2:
    def __init__(self, maxNumbers: int):
        self.mask = (1 << maxNumbers) - 1
    def get(self) -> int:
        if self.mask == 0:
            return -1
        low = self.mask & -self.mask
        self.mask ^= low
        return low.bit_length() - 1
    def check(self, number: int) -> bool:
        return (self.mask >> number) & 1 == 1
    def release(self, number: int) -> None:
        self.mask |= 1 << number

for _ in range(200):
    maxn = rng.choice([1, 2, 5, 12, 40])
    names, args = tao_lenh(rng, maxn, rng.randint(1, 60))
    kiem_tra_hop_le(PhoneDirectoryBits2, names, args)
    assert run_commands(PhoneDirectoryBits2, names, args) == run_commands(PhoneDirectoryScan, names, args)
print("Bài 4 OK")

# Bài 5
class SmallestInfiniteSet:
    def __init__(self):
        self.cur = 1                 # mọi số >= cur vẫn còn trong tập
        self.heap = []               # các số < cur đã bị pop rồi được thêm lại
        self.in_heap = set()

    def popSmallest(self) -> int:
        if self.heap:
            x = heapq.heappop(self.heap)
            self.in_heap.remove(x)
            return x
        x = self.cur
        self.cur += 1
        return x

    def addBack(self, num: int) -> None:
        if num < self.cur and num not in self.in_heap:
            heapq.heappush(self.heap, num)
            self.in_heap.add(num)

s = SmallestInfiniteSet()
assert [s.popSmallest() for _ in range(3)] == [1, 2, 3]
s.addBack(2)
assert s.popSmallest() == 2 and s.popSmallest() == 4
s.addBack(10)                         # 10 >= cur nên vốn còn trong tập -> không thêm
assert s.popSmallest() == 5

# So với mô hình vét cạn: tập con của 1..N đã bị pop
rng = random.Random(55)
for _ in range(300):
    s, daxoa = SmallestInfiniteSet(), set()
    for _ in range(rng.randint(1, 60)):
        if rng.random() < 0.6:
            x = s.popSmallest()
            m = 1
            while m in daxoa:
                m += 1
            assert x == m
            daxoa.add(x)
        else:
            num = rng.randint(1, 20)
            s.addBack(num)
            daxoa.discard(num)
print("Bài 5 OK")

Bài 1: [None, 0, 1, None, 2, 0, -1]
Bài 2: [None, 0, 1, 2, None, None, 1, 3, -1, -1]
lệnh                   kết quả  next_free  recycled   used
PhoneDirectory(4)      None     0          []         []
get()                  0        1          []         [0]
get()                  1        2          []         [0, 1]
get()                  2        3          []         [0, 1, 2]
release(1)             None     3          [1]        [0, 2]
release(1)             None     3          [1]        [0, 2]
get()                  1        3          []         [0, 1, 2]
get()                  3        4          []         [0, 1, 2, 3]
get()                  -1       4          []         [0, 1, 2, 3]
get()                  -1       4          []         [0, 1, 2, 3]
Bài 3 OK
Bài 4 OK
Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (quản lý tập tài nguyên bằng `set`):**

```python
class PhoneDirectory:
    def __init__(self, maxNumbers: int):
        self.available = set(range(maxNumbers))

    def get(self) -> int:
        if not self.available:
            return -1
        return self.available.pop()

    def check(self, number: int) -> bool:
        return number in self.available

    def release(self, number: int) -> None:
        self.available.add(number)
```

**Template (khởi tạo lười):** `next_free` + `recycled` (ngăn xếp) + `used` (set); `release` chỉ khi `number in used`.

**Checklist:**

- [ ] Mỗi số chỉ ở **một** trạng thái (trống/đã cấp); cấu trúc "trống" không chứa trùng.
- [ ] `get` **xóa** số khỏi cấu trúc trống (dùng `pop`/`popleft`) và trả `-1` khi hết.
- [ ] `release` idempotent: gọi lại với số đã trống không phá bất biến (set tự chống; hàng đợi/stack phải có kiểm tra cờ).
- [ ] `check` không thay đổi trạng thái; `release` trả `None`.
- [ ] Không sửa `set` khi đang duyệt; dùng `deque` (không dùng `list.pop(0)`) cho hàng đợi.
- [ ] Test bằng **mô hình tham chiếu** (get có thể trả số bất kỳ), không so thứ tự cứng.
- [ ] Đúng tên lớp `PhoneDirectory` và các method `get`, `check`, `release` (viết thường).
- [ ] Bài Premium: đối chiếu lại đề gốc nếu có tài khoản.